# Lesson 1: Web scraping basics with requests + BeautifulSoup

This tutorial goes from "download a page" to "scrape many pages and save
the data to CSV". Each section builds on the one before it.

Install the libraries first:

```bash
pip install -r requirements.txt   # from the repo's top folder
```

We'll mostly scrape **https://quotes.toscrape.com**, a site made for
practising web scraping, and then a real page on Wikipedia.

In [1]:
import csv
import json
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

OUTPUT = Path("output")  # every lesson saves its files here
OUTPUT.mkdir(exist_ok=True)

## 1. Download a page with `requests`

Three habits worth learning right away:

* **Send a User-Agent** so the site knows who's visiting. Many sites
  (Wikipedia included) block requests that don't send one.
* **Set a timeout** so your script doesn't hang forever.
* **Check the status code**. `raise_for_status()` raises an error for
  404, 500 and so on.

In [2]:
HEADERS = {
    "User-Agent": "web-scraping-with-python tutorial "
    "(+https://github.com/rajat4665/web-scraping-with-python)"
}

url = "https://quotes.toscrape.com/"
response = requests.get(url, headers=HEADERS, timeout=15)
response.raise_for_status()

print("Status code:", response.status_code)
print("Content type:", response.headers["Content-Type"])
print("First 200 characters of HTML:\n", response.text[:200])

Status code: 200
Content type: text/html; charset=utf-8
First 200 characters of HTML:
 <!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/m


## 2. Parse the HTML with BeautifulSoup

BeautifulSoup turns the raw HTML text into a tree you can search.
`"html.parser"` is built into Python, so it needs no extra install.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

# find() returns the FIRST matching tag, or None if there isn't one
title = soup.find("title")
print("With HTML tags:   ", title)
print("Without HTML tags:", title.get_text(strip=True))

With HTML tags:    <title>Quotes to Scrape</title>
Without HTML tags: Quotes to Scrape


## 3. `find()` vs `find_all()`

* `find()` returns the first match (or `None`)
* `find_all()` returns a list of every match

In [4]:
first_link = soup.find("a")
print("First link:", first_link)

all_links = soup.find_all("a")
print("Total links on the page:", len(all_links))

for link in all_links[:5]:  # slicing: just the first 5
    print(" ", link.get_text(strip=True), "->", link.get("href"))

First link: <a href="/" style="text-decoration: none">Quotes to Scrape</a>
Total links on the page: 55
  Quotes to Scrape -> /
  Login -> /login
  (about) -> /author/Albert-Einstein
  change -> /tag/change/page/1/
  deep-thoughts -> /tag/deep-thoughts/page/1/


## 4. Read attributes

Use `tag["attr"]` when you're sure the attribute exists, and
`tag.get("attr")` when it might not (it returns `None` instead of raising
a `KeyError`).

Links on a page are often *relative* (`/page/2/`). `urljoin` turns them
into full URLs.

In [5]:
for link in soup.find_all("a", href=True)[:5]:
    print(link["href"], "->", urljoin(url, link["href"]))

# `class` is special: BeautifulSoup gives it to you as a list,
# because an element can have several classes.
first_quote = soup.find("div", class_="quote")
print("\nclass attribute:", first_quote["class"], type(first_quote["class"]))
print("joined into a string:", " ".join(first_quote["class"]))

/ -> https://quotes.toscrape.com/
/login -> https://quotes.toscrape.com/login
/author/Albert-Einstein -> https://quotes.toscrape.com/author/Albert-Einstein
/tag/change/page/1/ -> https://quotes.toscrape.com/tag/change/page/1/
/tag/deep-thoughts/page/1/ -> https://quotes.toscrape.com/tag/deep-thoughts/page/1/

class attribute: ['quote'] <class 'bs4.element.AttributeValueList'>
joined into a string: quote


## 5. Search by class, and look *inside* an element

Calling `find()` on a tag searches only inside that tag, which is how you
pull structured data out of a page.

In [6]:
for quote in soup.find_all("div", class_="quote")[:3]:
    text = quote.find("span", class_="text").get_text(strip=True)
    author = quote.find("small", class_="author").get_text(strip=True)
    tags = [tag.get_text() for tag in quote.find_all("a", class_="tag")]
    print(f"{text}\n    - {author}  {tags}\n")

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
    - Albert Einstein  ['change', 'deep-thoughts', 'thinking', 'world']

“It is our choices, Harry, that show what we truly are, far more than our abilities.”
    - J.K. Rowling  ['abilities', 'choices']

“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
    - Albert Einstein  ['inspirational', 'life', 'live', 'miracle', 'miracles']



## 6. CSS selectors: `select()` and `select_one()`

If you know CSS, selectors are often shorter than chains of `find()`
calls. Here's the same data again:

| Selector            | Means                                   |
|---------------------|-----------------------------------------|
| `div.quote`         | `<div>` with class `quote`              |
| `span.text`         | `<span>` with class `text`              |
| `div.tags a.tag`    | `<a class="tag">` inside `<div class="tags">` |
| `li.next > a`       | an `<a>` directly inside `<li class="next">`  |

In [7]:
for quote in soup.select("div.quote")[:3]:
    print(quote.select_one("span.text").get_text(strip=True))
    print("   by", quote.select_one("small.author").get_text(strip=True))
    print("   tags:", [a.get_text() for a in quote.select("div.tags a.tag")])

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
   by Albert Einstein
   tags: ['change', 'deep-thoughts', 'thinking', 'world']
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
   by J.K. Rowling
   tags: ['abilities', 'choices']
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
   by Albert Einstein
   tags: ['inspirational', 'life', 'live', 'miracle', 'miracles']


## 7. Scrape every page (pagination)

Most sites split data across pages. Keep following the "Next" link until
there isn't one. `time.sleep()` between requests keeps the load on the
server low. Please be polite!

In [8]:
def parse_quotes(page_soup):
    """Return a list of dicts, one per quote on the page."""
    return [
        {
            "text": q.select_one("span.text").get_text(strip=True),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags": [a.get_text() for a in q.select("a.tag")],
        }
        for q in page_soup.select("div.quote")
    ]


def scrape_all_quotes(start_url, max_pages=10, delay=0.5):
    session = requests.Session()  # reuses the connection: faster and kinder
    session.headers.update(HEADERS)

    quotes, page_url, pages = [], start_url, 0
    while page_url and pages < max_pages:
        page = session.get(page_url, timeout=15)
        page.raise_for_status()
        page_soup = BeautifulSoup(page.text, "html.parser")

        quotes.extend(parse_quotes(page_soup))
        pages += 1
        print(f"page {pages}: {page_url} ({len(quotes)} quotes so far)")

        next_link = page_soup.select_one("li.next > a")
        page_url = urljoin(page_url, next_link["href"]) if next_link else None
        time.sleep(delay)
    return quotes


all_quotes = scrape_all_quotes("https://quotes.toscrape.com/")
print("\nTotal quotes scraped:", len(all_quotes))

page 1: https://quotes.toscrape.com/ (10 quotes so far)


page 2: https://quotes.toscrape.com/page/2/ (20 quotes so far)


page 3: https://quotes.toscrape.com/page/3/ (30 quotes so far)


page 4: https://quotes.toscrape.com/page/4/ (40 quotes so far)


page 5: https://quotes.toscrape.com/page/5/ (50 quotes so far)


page 6: https://quotes.toscrape.com/page/6/ (60 quotes so far)


page 7: https://quotes.toscrape.com/page/7/ (70 quotes so far)


page 8: https://quotes.toscrape.com/page/8/ (80 quotes so far)


page 9: https://quotes.toscrape.com/page/9/ (90 quotes so far)


page 10: https://quotes.toscrape.com/page/10/ (100 quotes so far)



Total quotes scraped: 100


## 8. Save the data (CSV and JSON)

In [9]:
with open(OUTPUT / "quotes.csv", "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["text", "author", "tags"])
    writer.writeheader()
    for q in all_quotes:
        writer.writerow({**q, "tags": ", ".join(q["tags"])})

with open(OUTPUT / "quotes.json", "w", encoding="utf-8") as fh:
    json.dump(all_quotes, fh, indent=2, ensure_ascii=False)

print("Saved output/quotes.csv and output/quotes.json")

Saved output/quotes.csv and output/quotes.json


## 9. A real website: Wikipedia

Real sites are messier. Some tips:

* Open the page in your browser, right-click and choose **Inspect** to find
  the tags and classes you need.
* Websites change their HTML over time, so selectors that worked last year
  may not work today. Always check for `None`.

For example, this is how Wikipedia's table of contents looked in DevTools
when this tutorial was first written. It was a `<div id="toc" class="toc">`.
That element no longer exists, so code that relied on it now finds nothing.

![Wikipedia's old table of contents in DevTools](../assets/devtools-inspect-wikipedia.png)

In [10]:
wiki_url = "https://en.wikipedia.org/wiki/World_War_II"
wiki = requests.get(wiki_url, headers=HEADERS, timeout=15)
wiki.raise_for_status()
wiki_soup = BeautifulSoup(wiki.text, "html.parser")

print("Title:", wiki_soup.find("title").get_text(strip=True))

# Section headings
headings = [h.get_text(strip=True) for h in wiki_soup.select("div.mw-heading2 h2")]
print("\nSections:", headings)

Title: World War II - Wikipedia

Sections: ['Start and end dates', 'Background', 'Pre-war events', 'Course of the war', 'Aftermath', 'Impact', 'See also', 'Notes', 'References', 'Further reading', 'External links']


### Turn the infobox table into a dictionary

The infobox is the summary box at the top right of a Wikipedia article.
Each row has a label (`<th>`) and a value (`<td>`).

In [11]:
infobox = wiki_soup.select_one("table.infobox")
facts = {}
if infobox is None:
    print("No infobox found. Wikipedia's HTML may have changed.")
else:
    for row in infobox.find_all("tr"):
        label, value = row.find("th"), row.find("td")
        if label and value:
            # remove citation markers like [1]
            for sup in value.find_all("sup", class_="reference"):
                sup.decompose()
            facts[label.get_text(" ", strip=True)] = value.get_text(" ", strip=True)

for key, value in list(facts.items())[:6]:
    print(f"{key}: {value[:100]}")

Date: 1 September 1939 – 2 September 1945 (6 years, 1 day)
Location: Global
Result: Allied victory (see Aftermath of World War II )


### Images

Image `src` values on Wikipedia start with `//`, so `urljoin` is useful
again.

In [12]:
images = [urljoin(wiki_url, img["src"]) for img in wiki_soup.find_all("img", src=True)]
print("Images found:", len(images))
for src in images[:5]:
    print(" ", src)

Images found: 67
  https://en.wikipedia.org/static/images/icons/enwiki-25.svg
  https://en.wikipedia.org/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg
  https://en.wikipedia.org/static/images/mobile/copyright/wikipedia-tagline-en-25.svg
  https://thumb.wikimedia.org/wikipedia/en/thumb/9/94/Symbol_support_vote.svg/20px-Symbol_support_vote.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail
  https://thumb.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail


## Scraping responsibly

* Check the site's `robots.txt` (e.g. https://en.wikipedia.org/robots.txt)
  and its terms of service.
* If the site has an API (Wikipedia does), prefer it.
* Add a delay between requests and don't hammer servers.
* Don't scrape personal data, or content behind a login, without permission.

**Next:** [Lesson 2](02_hidden_apis_and_embedded_json.ipynb): hidden APIs and
embedded JSON. Before writing selectors, check whether the data is already
available as JSON.

Also try the `webscraper` command line tool in this repo
(`python -m webscraper`), which wraps these ideas into a reusable tool.